In [2]:
import json, time
from collections import Counter
import pandas as pd
from confluent_kafka import Consumer

consumer = Consumer({"bootstrap.servers": "localhost:9092,localhost:9094,localhost:9096",
                      "group.id": "urbanpulse-dlq-report", "auto.offset.reset": "earliest"})
consumer.subscribe(["urbanpulse.dlq"])

counts = Counter()
end = time.time() + 300     # 5 minutes
while time.time() < end:
    msg = consumer.poll(1.0)
    if msg is None or msg.error():
        continue
    event = json.loads(msg.value())
    counts[event.get("error_reason", "unknown")] += 1

df = pd.DataFrame(counts.items(), columns=["error_reason", "count"]).sort_values("count", ascending=False)
df["pct"] = (df["count"] / df["count"].sum() * 100).round(1)
print(df)
df.to_csv("dlq_5min_report.csv", index=False)

       error_reason  count   pct
2       invalid_gps   1924  85.5
0        null_value    314  14.0
1  out_of_range_aqi     12   0.5
